# Practice Lab 5.5 - MCP Protocol Fundamentals (WORKED solutions)

**Course:** Netsetos FDE Cohort 1 - Module 5 - Lesson 5.5 - Practice Lab

This notebook is the **worked-solution companion** to `practice-lab-5.5.html` (9 exercises,
160 points). Each exercise below repeats the prompt and success criteria, then shows a runnable
worked solution. Run it top to bottom.

- **Everything runs offline - no API key is required for any exercise.** MCP is a client-to-server
  JSON-RPC protocol, so the graded artifact is the protocol shape, not a model call. The keyless
  guard in the next cell still mocks the `anthropic` / `openai` clients so any illustrative model
  call executes end to end.
- Production context: you are learning the protocol against **AcmeCorp's CRM server** - the
  artifact L5.6 builds for real. Synthetic customer IDs (`c_4823`) and ticket IDs (`T9201`) only.
- Runnable Python also ships as `solutions/exercise_1_mcp_client_call.py` (real stdio client) and
  `solutions/exercise_2_acmecorp_crm_mcp_consumer.py` (in-process mock consumer).


In [ ]:
!pip install --quiet anthropic openai


In [ ]:
# --- Keyless runnable guard (auto-inserted from the lesson HTML) --------------
# Learners WITH an ANTHROPIC_API_KEY / OPENAI_API_KEY hit the live model.
# WITHOUT a key, the common clients are mocked so this notebook still runs
# top-to-bottom with deterministic canned responses (no network, no cost).
import os
_HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
_HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
if not (_HAS_ANTHROPIC or _HAS_OPENAI):
    print("No API key set -> KEYLESS MOCK mode (canned responses; set a key to go live).")
    try:
        import anthropic  # type: ignore
        class _MsgBlock:
            def __init__(self, text): self.text = text; self.type = "text"
        class _Msg:
            def __init__(self, text):
                self.content = [_MsgBlock(text)]
                self.stop_reason = "end_turn"
                self.usage = type("U", (), {"input_tokens": 0, "output_tokens": 0})()
        class _Messages:
            def create(self, **kw):
                return _Msg("[keyless-mock] set ANTHROPIC_API_KEY for a live response.")
        class _Anthropic:
            def __init__(self, *a, **k): self.messages = _Messages()
        anthropic.Anthropic = _Anthropic
        if hasattr(anthropic, "AsyncAnthropic"): anthropic.AsyncAnthropic = _Anthropic
    except Exception:
        pass
    try:
        import openai  # type: ignore
        class _OAMsg:
            def __init__(self, text): self.content = text
        class _OAChoice:
            def __init__(self, text): self.message = _OAMsg(text)
        class _OAResp:
            def __init__(self, text): self.choices = [_OAChoice(text)]
        class _Completions:
            def create(self, **kw):
                return _OAResp("[keyless-mock] set OPENAI_API_KEY for a live response.")
        class _Chat:
            def __init__(self): self.completions = _Completions()
        class _OpenAI:
            def __init__(self, *a, **k): self.chat = _Chat()
        openai.OpenAI = _OpenAI
        if hasattr(openai, "AsyncOpenAI"): openai.AsyncOpenAI = _OpenAI
    except Exception:
        pass
# ----------------------------------------------------------------------------


---
## Exercise 1 (12 pts) - Label the role: host, client, or server?

**Task.** For each of the six named systems, write `host`, `client`, or `server` plus one
sentence of justification. The host owns the UI and the conversation, the client is the
per-server connection manager inside the host, and the server is the out-of-process program
advertising tools / resources / prompts.

**Success criteria**
- All six roles correct: 1 host, 2 server, 3 host, 4 server, 5 client, 6 server.
- Each label carries a one-line justification (the client-vs-host distinction is the trap).


In [ ]:
# Worked solution - the role for each system, with a one-line justification.
ROLES = {
    "1. Claude Desktop":                     ("host",   "owns the UI + conversation; embeds MCP clients"),
    "2. filesystem reference server":        ("server", "out-of-process program advertising tools/resources"),
    "3. Cursor's MCP integration":           ("host",   "an IDE host that manages a client per connected server"),
    "4. AcmeCorp CRM service (L5.6)":         ("server", "advertises lookup_customer over JSON-RPC"),
    "5. Python script using ClientSession":  ("client", "the per-server connection manager, even with no host UI"),
    "6. GitHub's official MCP server":        ("server", "third-party server exposing GitHub tools"),
}
EXPECTED = ["host", "server", "host", "server", "client", "server"]
assert [role for role, _ in ROLES.values()] == EXPECTED
for name, (role, why) in ROLES.items():
    print(f"{role:7s} | {name} - {why}")
print("OK - 1 host, 2 server, 3 host, 4 server, 5 client, 6 server.")


---
## Exercise 2 (15 pts) - Connect to a stdio server: list, call, print

**Task.** Complete the connect-list-call pattern against an MCP server: `list_tools()`, confirm
the tool is advertised, `call_tool(...)`, return the content blocks + `isError`. This is the bare
protocol surface - no host UI, no LLM, just client and server exchanging JSON-RPC.

The real stdio version is `solutions/exercise_1_mcp_client_call.py` (it imports `mcp` and speaks
JSON-RPC over stdin/stdout). To keep this **offline and keyless**, the worked solution below mocks
the server in-process - the identical list-then-call shape, no `mcp` package required.

**Success criteria**
- The client prints the advertised tool catalog, then the structured result of one `tools/call`.
- Content blocks + `isError` are returned; the call is confirmed against the advertised catalog.


In [ ]:
# Worked solution - the connect-list-call shape against an in-process server.
# Real version: solutions/exercise_1_mcp_client_call.py (stdio + JSON-RPC).
import json
from dataclasses import dataclass, field

@dataclass
class InProcessServer:
    name: str = "acmecorp-crm"
    call_log: list = field(default_factory=list)

    def list_tools(self):
        return [{
            "name": "lookup_customer",
            "description": "Fetch an AcmeCorp customer by id",
            "input_schema": {"type": "object",
                             "properties": {"customer_id": {"type": "string"}},
                             "required": ["customer_id"]},
        }]

    def call_tool(self, name, arguments):
        self.call_log.append({"name": name, "arguments": arguments})
        if name != "lookup_customer":
            return {"isError": True, "content": [{"type": "text", "text": f"unknown tool: {name}"}]}
        rec = {"customer_id": arguments["customer_id"], "name": "Acme Subsidiary Inc",
               "tier": "pro", "mrr_usd": 1299}
        return {"isError": False, "content": [{"type": "text", "text": json.dumps(rec)}]}

def list_and_call(server, tool_name, arguments):
    catalog = [t["name"] for t in server.list_tools()]          # tools/list
    print(f"server advertises {len(catalog)} tool(s): {catalog}")
    if tool_name not in catalog:
        raise ValueError(f"{tool_name!r} not in catalog: {catalog}")
    result = server.call_tool(tool_name, arguments)             # tools/call
    return result["content"], bool(result["isError"])

server = InProcessServer()
content, is_error = list_and_call(server, "lookup_customer", {"customer_id": "c_4823"})
print("isError:", is_error)
print("content:", content)
assert is_error is False
assert server.call_log[-1]["arguments"]["customer_id"] == "c_4823"
print("OK - listed the catalog, then called one tool (the connect-list-call shape).")


---
## Exercise 3 (20 pts) - Hand-trace the JSON-RPC wire

**Task.** AcmeCorp's CRM server advertises one tool, `lookup_customer(customer_id)`. Construct the
four JSON-RPC 2.0 envelopes that form a valid `tools/list` round-trip and a `tools/call` round-trip
for customer `c_4823`, then answer the two follow-ups.

**Success criteria**
- Both method strings correct (`tools/list`, `tools/call`).
- Arguments routed under `params.arguments`; `params.name` selects the tool.
- Result shape is a `content` list + `isError`.
- Follow-up 1: the two `id` fields that MUST match, and why.
- Follow-up 2: the three things the 2026-07-28 stateless spec changed.


In [ ]:
# Worked solution - the four envelopes, plus self-checks on the id-matching rule.
import json

# (A) tools/list REQUEST   (client -> server)
list_req  = {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}

# (B) tools/list RESPONSE  (server -> client)
list_resp = {"jsonrpc": "2.0", "id": 1, "result": {"tools": [{
    "name": "lookup_customer",
    "description": "Look up an AcmeCorp customer by id",
    "inputSchema": {"type": "object",
                    "properties": {"customer_id": {"type": "string"}},
                    "required": ["customer_id"]},
}]}}

# (C) tools/call REQUEST   (client -> server)
call_req  = {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
             "params": {"name": "lookup_customer", "arguments": {"customer_id": "c_4823"}}}

# (D) tools/call RESPONSE  (server -> client)
call_resp = {"jsonrpc": "2.0", "id": 2, "result": {
    "content": [{"type": "text", "text": json.dumps(
        {"customer_id": "c_4823", "name": "Acme Subsidiary", "tier": "pro", "mrr_usd": 1299})}],
    "isError": False}}

# Follow-up 1: a response is correlated to its request purely by matching id.
assert list_req["id"] == list_resp["id"]        # 1 == 1
assert call_req["id"] == call_resp["id"]         # 2 == 2
assert list_req["id"] != call_req["id"]          # the two round-trips use different ids
# arguments live under params.arguments, never at the top of params:
assert "arguments" in call_req["params"] and "name" in call_req["params"]

for label, env in [("A) tools/list REQUEST", list_req), ("B) tools/list RESPONSE", list_resp),
                   ("C) tools/call REQUEST", call_req), ("D) tools/call RESPONSE", call_resp)]:
    print(f"# {label}")
    print(json.dumps(env))
    print()

print("Follow-up 1: A.id must equal B.id and C.id must equal D.id - JSON-RPC correlates a")
print("             response to its request purely by matching id, which is how concurrent")
print("             calls on one transport get demultiplexed. Different round-trips use ids 1 and 2.")
FOLLOWUP_2 = [
    "no session id - each request is self-contained, so it can land on ANY instance behind a load balancer",
    "per-request capabilities travel in the message _meta field (not a once-negotiated initialize handshake)",
    "server/discover performs protocol-version negotiation on demand (initialize no longer does it at start)",
]
print("Follow-up 2 - what the 2026-07-28 stateless spec changed:")
for change in FOLLOWUP_2:
    print(" -", change)


---
## Exercise 4 (15 pts) - Pick the right primitive: tool, resource, or prompt

**Task.** Assign each of the five AcmeCorp capabilities to **tool**, **resource**, or **prompt**
and justify in one sentence. The test is who is deciding: the model (tool), the host attaching
context up-front (resource), or the user invoking by name (prompt).

**Success criteria**
- a=tool, b=resource, c=prompt, d=tool, e=resource.
- Justifications name *who decides* - e.g. a resource never burns a tool-call round-trip.


In [ ]:
# Worked solution - each capability mapped to the primitive whose invocation surface fits.
ASSIGN = {
    "a. create_ticket(customer_id, summary)":        ("tool",     "model decides to create -> model-invoked action"),
    "b. customer_history feed (last 20)":            ("resource", "host pre-attaches read-only context; no tool round-trip"),
    "c. /start-triage ticket_id=T9201":              ("prompt",   "user invokes the workflow by name from the host UI"),
    "d. refund_invoice(invoice_id)":                 ("tool",     "model-invoked action"),
    "e. compliance_policy_doc (read-only)":          ("resource", "static read-only context the host attaches up-front"),
}
assert [prim for prim, _ in ASSIGN.values()] == ["tool", "resource", "prompt", "tool", "resource"]
for cap, (prim, why) in ASSIGN.items():
    print(f"{prim:9s} | {cap} - {why}")
print("OK - who decides? model=tool, host=resource, user=prompt.")


---
## Exercise 5 (15 pts) - Match the interaction primitive: sampling, roots, or elicitation?

**Task.** Tools / resources / prompts flow client-to-server. The three *interaction primitives*
flow the other way - the server sends a request up to the client/host. **Sampling** lets a server
ask the host to run an LLM completion on its behalf. **Roots** let the client declare which
filesystem / URI boundaries a server may operate within. **Elicitation** lets a server pause
mid-operation to collect additional structured human input. Match each AcmeCorp scenario.

**Success criteria**
- a=elicitation, b=roots, c=sampling.
- The justification names the discriminator: the *server* initiates a request toward the host.


In [ ]:
# Worked solution - each scenario matched to the interaction primitive it needs.
INTERACTION = {
    "a. bulk_refund needs the human to confirm total + reason code mid-call": (
        "elicitation", "server pauses to collect structured human input before it can safely proceed"),
    "b. document-indexer may read only /acme/contracts and /acme/tickets": (
        "roots", "client declares the permitted filesystem/URI boundary the server cannot wander outside"),
    "c. triage server drafts a reply using the host's model, shipping no key of its own": (
        "sampling", "server asks the host to run a model completion on its behalf"),
}
assert [prim for prim, _ in INTERACTION.values()] == ["elicitation", "roots", "sampling"]
for scenario, (prim, why) in INTERACTION.items():
    print(f"{prim:12s} | {scenario}")
    print(f"{'':12s} | -> {why}")
print("Discriminator: in all three the SERVER initiates a request toward the host - the signature")
print("of an interaction primitive, which is exactly why none of these is a plain tool call.")


---
## Exercise 6 (18 pts) - Extend the AcmeCorp consumer

**Task.** Extend the mock CRM server: add a second tool `list_open_tickets(customer_id)` to
`list_tools()` and `call_tool()`, translate it into the Anthropic tool catalog, and drive a
2-step conversation (look up `c_4823`, then list their open tickets). Runs offline - see
`solutions/exercise_2_acmecorp_crm_mcp_consumer.py` for the full consumer.

**Success criteria**
- The Anthropic catalog carries both tools.
- Two routed tool calls; the server call log shows 2 entries.
- `c_4823` (Acme Subsidiary, pro, $1299 MRR) resolves and `list_open_tickets("c_4823")` returns 1 open ticket.


In [ ]:
# Worked solution - the extended mock server + a 2-step conversation (offline).
import json, time
from dataclasses import dataclass, field

_MOCK_CUSTOMERS = {
    "c_4823": {"customer_id": "c_4823", "name": "Acme Subsidiary Inc",
               "tier": "pro", "mrr_usd": 1299, "open_tickets": 1},
}
_MOCK_TICKETS = {
    "c_4823": [{"ticket_id": "T9201", "status": "open", "subject": "Invoice question"}],
}

@dataclass
class MockMCPServer:
    name: str = "acmecorp-crm"
    call_log: list = field(default_factory=list)

    def list_tools(self):
        return [
            {"name": "lookup_customer",
             "description": "Fetch an AcmeCorp customer record by ID",
             "input_schema": {"type": "object",
                              "properties": {"customer_id": {"type": "string", "pattern": "^c_[a-z0-9]+$"}},
                              "required": ["customer_id"]}},
            # --- Exercise 6: the second tool you add ---
            {"name": "list_open_tickets",
             "description": "List open support tickets for an AcmeCorp customer",
             "input_schema": {"type": "object",
                              "properties": {"customer_id": {"type": "string", "pattern": "^c_[a-z0-9]+$"}},
                              "required": ["customer_id"]}},
        ]

    def call_tool(self, name, arguments):
        self.call_log.append({"name": name, "arguments": arguments, "at": time.time()})
        cid = arguments.get("customer_id", "")
        if name == "lookup_customer":
            rec = _MOCK_CUSTOMERS.get(cid)
            if rec is None:
                return {"isError": True, "content": [{"type": "text", "text": f"not found: {cid}"}]}
            return {"isError": False, "content": [{"type": "text", "text": json.dumps(rec)}]}
        if name == "list_open_tickets":
            tix = [t for t in _MOCK_TICKETS.get(cid, []) if t["status"] == "open"]
            return {"isError": False, "content": [{"type": "text", "text": json.dumps(tix)}]}
        return {"isError": True, "content": [{"type": "text", "text": f"unknown tool: {name}"}]}

def to_anthropic_tool(mcp_tool):
    """The exact translation a real host's MCP client does under the hood."""
    return {"name": mcp_tool["name"], "description": mcp_tool["description"],
            "input_schema": mcp_tool["input_schema"]}

server = MockMCPServer()
anthropic_tools = [to_anthropic_tool(t) for t in server.list_tools()]
print("Anthropic catalog:", [t["name"] for t in anthropic_tools])

# 2-step conversation (offline: we simulate the model choosing each tool_use in turn).
step1 = server.call_tool("lookup_customer", {"customer_id": "c_4823"})
print("step 1  lookup_customer   ->", step1["content"][0]["text"])
step2 = server.call_tool("list_open_tickets", {"customer_id": "c_4823"})
print("step 2  list_open_tickets ->", step2["content"][0]["text"])

assert len(anthropic_tools) == 2
assert len(server.call_log) == 2
tickets = json.loads(step2["content"][0]["text"])
assert len(tickets) == 1 and tickets[0]["ticket_id"] == "T9201"
print(f"OK - 2 routed tool calls; call log shows {len(server.call_log)} entries; 1 open ticket.")


---
## Exercise 7 (25 pts) - Harden a remote MCP server: the four threats

**Task.** AcmeCorp is about to expose its CRM MCP server as a **remote streamable-HTTP** service
(OAuth 2.1, 200 employees, reachable from Claude.ai web). Write the single most load-bearing
mitigation for each of the four canonical MCP threats.

**Success criteria**
- Confused deputy, token passthrough, tool poisoning, and rug pull each named and closed with a
  mitigation that actually holds.
- Bonus: ground at least two mitigations in the specific mechanism (identity propagation /
  audience restriction, token exchange vs passthrough, Enterprise Managed Authorization, catalog
  hashing + re-consent on `list_changed`).


In [ ]:
# Worked solution - one load-bearing mitigation per threat (the graded payload).
MITIGATIONS = {
    "confused_deputy":
        "propagate the END USER's identity to the CRM (not one shared service-account credential) "
        "and enforce per-user authorization at the tool boundary; audience-restrict tokens "
        "(RFC 8707 resource indicators, RFC 9207 issuer identification) so a token minted for one "
        "resource cannot be replayed at another.",
    "token_passthrough":
        "validate aud / issuer / signature on EVERY request (the stateless core means validate per "
        "request, not once), reject any token whose audience is not this server, and never forward "
        "the client's token downstream - exchange it (RFC 8693) for a correctly scoped token.",
    "tool_poisoning":
        "treat third-party tool descriptions and schemas as untrusted data, not instructions; "
        "allowlist / pin vetted servers via Enterprise Managed Authorization, surface tool "
        "descriptions to a human on install, and keep AcmeCorp's CRM server first-party so its "
        "descriptions are code-reviewed rather than fetched blind.",
    "rug_pull":
        "version-pin the server and hash or sign its tool catalog; do NOT silently honor "
        "notifications/tools/list_changed - require re-consent when a tool's description or schema "
        "hash changes, and log plus alert on catalog drift.",
}
assert set(MITIGATIONS) == {"confused_deputy", "token_passthrough", "tool_poisoning", "rug_pull"}
for threat, fix in MITIGATIONS.items():
    print(f"[{threat}]")
    print(f"  {fix}\n")
print("Every one of these threats gets WORSE, not better, once the server is remote and multi-tenant.")


---
## Exercise 8 (20 pts) - Transport decision matrix

**Task.** An AcmeCorp customer must choose one transport for their CRM MCP server; 200 employees
reach it through Claude.ai web. Fill every cell for **stdio**, **deprecated HTTP+SSE**, and
**streamable HTTP**, then defend the winner in two sentences.

**Success criteria**
- 15 cells filled correctly.
- Streamable HTTP chosen; the 2-sentence justification cites stateless-core scaling *and* rules
  out both losers by their disqualifier.


In [ ]:
# Worked solution - the filled matrix + the verdict, as checkable data.
TRANSPORTS = {
    "stdio": {
        "spec_status": "current (local transport)",
        "reach": "local only (subprocess)",
        "load_balancer": "N/A (no network)",
        "resumable": "N/A",
        "cdn": "no",
        "verdict": "reject (local only)"},
    "http_sse": {
        "spec_status": "deprecated, 12-month offramp",
        "reach": "remote",
        "load_balancer": "no - sticky per-session SSE",
        "resumable": "no",
        "cdn": "poorly",
        "verdict": "reject (deprecated)"},
    "streamable_http": {
        "spec_status": "current default (remote)",
        "reach": "remote",
        "load_balancer": "yes - stateless core",
        "resumable": "yes - Last-Event-ID",
        "cdn": "yes - single endpoint",
        "verdict": "choose"},
}
rows = ["spec_status", "reach", "load_balancer", "resumable", "cdn", "verdict"]
hdr = f"{'criterion':14s} | " + " | ".join(f"{t:22s}" for t in TRANSPORTS)
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(f"{r:14s} | " + " | ".join(f"{TRANSPORTS[t][r]:22s}" for t in TRANSPORTS))

winner = [t for t, v in TRANSPORTS.items() if v["verdict"] == "choose"]
assert winner == ["streamable_http"]
print("\nVerdict: streamable HTTP is the only current remote transport that stays a single endpoint")
print("with a stateless core, so any instance behind the load balancer can serve any of the 200")
print("users and the whole thing sits behind a CDN. stdio is a local subprocess with no network")
print("reach, and HTTP+SSE is on the 12-month deprecation offramp with sticky sessions that fall")
print("over behind a load balancer.")


---
## Exercise 9 (20 pts) - Auth decision matrix

**Task.** For three customer profiles, pick the recommended pattern (**OAuth 2.1 + PKCE**,
**API key in `Authorization: Bearer`**, or **mTLS**), give a one-line reason, and name the one
objection you would rebut in the room.

**Success criteria**
- SaaS startup -> OAuth 2.1 + PKCE; zero-trust bank -> mTLS; internal-only -> API key.
- Each row carries a reason and a rebutted objection.


In [ ]:
# Worked solution - auth matched to posture, with the objection to rebut.
AUTH = {
    "1. SaaS startup (users have own accounts)": {
        "pattern": "OAuth 2.1 + PKCE",
        "reason": "per-user revocable tokens; PKCE blocks auth-code interception",
        "rebut": "'consent UX adds friction' - true, but it is the spec-preferred profile, "
                 "hardened by RFC 9207 + Client ID Metadata Documents"},
    "2. Fortune-500 bank (zero-trust, no shared secrets)": {
        "pattern": "mTLS",
        "reason": "certs authenticate at the TLS handshake - no long-lived shared secret to leak; "
                  "pairs with the service mesh",
        "rebut": "'high PKI / rotation cost' - that cost is the price of the zero-trust mandate"},
    "3. Internal AcmeCorp-only (no per-end-user identity)": {
        "pattern": "API key in Authorization: Bearer",
        "reason": "fine for v0 service-to-service where there is no per-end-user identity",
        "rebut": "'no rotation story' - plan rotation up front, and never put the key in a URL "
                 "query string (it leaks into logs)"},
}
assert [v["pattern"] for v in AUTH.values()] == [
    "OAuth 2.1 + PKCE", "mTLS", "API key in Authorization: Bearer"]
for profile, v in AUTH.items():
    print(f"[{profile}]")
    print(f"  pattern: {v['pattern']}")
    print(f"  reason : {v['reason']}")
    print(f"  rebut  : {v['rebut']}\n")
print("OK - auth matched to posture across three profiles.")


---
## Grading rubric + submission checklist

| Exercise | Points | Full marks |
|---|---|---|
| 1. Role labelling | 12 | Six roles correct with a one-line justification; client-vs-host distinction clear |
| 2. stdio connect-list-call | 15 | `list_tools` then `call_tool` wired; content blocks + `isError` returned; keyless fallback used cleanly |
| 3. JSON-RPC hand-trace | 20 | Four envelopes valid; arguments under `params.arguments`; id-matching explained; three stateless changes named |
| 4. Primitive selection | 15 | Five mapped to tool/resource/prompt with the "who decides" justification |
| 5. Interaction primitive | 15 | sampling / roots / elicitation matched; server-to-host direction named as the discriminator |
| 6. Consumer extension | 18 | `list_open_tickets` added to server + Anthropic catalog; 2-step conversation; call log shows 2 entries |
| 7. Four-threat hardening | 25 | Confused-deputy / token-passthrough / tool-poisoning / rug-pull each named and closed |
| 8. Transport decision matrix | 20 | 15 cells correct; streamable HTTP chosen; justification cites stateless scaling + both disqualifiers |
| 9. Auth decision matrix | 20 | All three profiles matched (OAuth / mTLS / API key) with reason + one objection rebutted |

**Pass mark: 112/160 (70%).** A distinction (144+, 90%) also names the anti-example correctly (a
hardcoded `client.messages.create(..., tools=[...])` call is function calling, not MCP - no server,
no JSON-RPC wire, no capability discovery), explains why the 2026-07-28 spec dropped the stateful
`initialize` handshake to make MCP horizontally scalable, and can state why every threat in
Exercise 7 gets *worse*, not better, once the server is remote and multi-tenant.

**Feedback SLA** - submit your solution in the cohort Discord `#practice-labs` channel;
rubric-graded feedback is returned within **24 hours** (Plus-tier mentor review). Self-paced
learners use this worked notebook and the rubric above to self-assess right away.
